# Projeto 1 – Método da Agulha de Buffon (estimação de π por Monte Carlo)

- Este programa implementa a simulação do lançamento de uma agulha sobre um piso com tábuas paralelas, estima o valor de π e analisa a convergência para diferentes números de lançamentos.

Autor: Lucas Brasil de Cerqueira

Data: 09/09/2026

## 1. Importando as bibliotecas

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import time

## 2. Definindo parâmetros

In [2]:
L = 1.0          # comprimento da agulha
D = 1.0          # largura das tábuas

## 3. Verificando a garantia da condição incial

In [3]:
if L > D:
    raise ValueError("O comprimento da agulha deve ser menor ou igual à largura das tábuas.")

## 4. Definindo funções auxiliares

### 4.1 Função: um_lancamento(l, d)

Simula um único lançamento da agulha.

#### Parâmetros:
- l (comprimento da agulhas) 
- d (distância entre linhas)

#### Retorna:
True se a agulha cruzar uma linha, False caso contrário.

In [4]:
def um_lancamento(l, d):
    
    # x: distância perpendicular de uma extremidade até a linha mais próxima
    # theta: ângulo agudo entre a agulha e a linha (0 a pi)
    
    x = np.random.uniform(0, d)
    theta = np.random.uniform(0, np.pi)
    
    # Condição de cruzamento (extremidade da agulha toca a linha)
    return x <= l * np.sin(theta)

### 4.2 Função: experimento(N, l, d)

Realiza N lançamentos e calcula a estimativa de π.

#### Parâmetros:
- N (número de lançamentos)
- l (comprimento da agulha)
- d (distância entre as linhas)

#### Retorna:
valor estimado de π (float)

In [5]:
def experimento(N, l, d):
  
    cruzamentos = 0
    for _ in range(N):
        if um_lancamento(l, d):
            cruzamentos += 1
            
    # Probabilidade empírica
    P = cruzamentos / N
    
    # Estimativa de π
    pi_est = (2.0*l)/(P*d)
    return pi_est

### 4.3 Função: repeticoes(N, M, l, d)

Repete o experimento M vezes para um dado N.

#### Parâmetros:
- N (número de lançamentos)
- M (número de repetições)
- l (comprimento da agulha)
- d (distância entre as linhas)

#### Retorna:
lista com M estimativas de π.

In [6]:
def repeticoes(N, M, l, d):
    
    estimativas = []
    for _ in range(M):
        estimativas.append(experimento(N, l, d))
    return estimativas

## 5. Simulações para diferentes números de lançamento

In [ ]:
# Números de lançamentos a serem testados
N_values = [100, 1000, 10000]
           
# Número de repetições para cada N 
M_values = [1000, 1000, 10000]  
           
# Dicionário para armazenar os resultados
resultados = {}

# Medição do tempo total de execução
tempo_total = time.time()

# Loop sobre os valores
for N, M in zip(N_values, M_values):
    print(f"Executando N = {N:5d} lançamentos com {M:5d} repetições...", end=" ")
    inicio = time.time()
    estimativas = repeticoes(N, M, L, D)
    fim = time.time()
    print(f"concluído em {fim - inicio:.2f} s")
    
    
    media = np.mean(estimativas)
    desvio = np.std(estimativas, ddof=1)  
    
    resultados[N] = {
        'M': M,
        'estimativas': estimativas,
        'media': media,
        'desvio': desvio
    }
    print(f"  Média = {media:.6f}, Desvio = {desvio:.6f}")
    
tempo_total = time.time() - tempo_total
print(f"\nTempo total de execução: {tempo_total:.2f} s\n")


Executando N =   100 lançamentos com  1000 repetições... concluído em 0.66 s
  Média = 3.157847, Desvio = 0.242334
Executando N =  1000 lançamentos com  1000 repetições... concluído em 6.65 s
  Média = 3.140839, Desvio = 0.072392
Executando N = 10000 lançamentos com 10000 repetições... 

## 6. Resultados

In [ ]:
print("=" * 70)
print("RESULTADOS DAS SIMULAÇÕES")
print("=" * 70)
print(f"{'N (lançamentos)':>15} | {'M (repetições)':>15} | {'Média de π':>15} | {'Desvio padrão':>15}")
print("-" * 70)
for N in N_values:
    med = resultados[N]['media']
    des = resultados[N]['desvio']
    M = resultados[N]['M']
    print(f"{N:15d} | {M:15d} | {med:15.6f} | {des:15.6f}")
print("=" * 70)
print(f"Valor real de π ≈ {np.pi:.6f}")
print("=" * 70)

## 7. Histograma e ajuste

In [ ]:
N_extra = 10000
if N_extra in resultados:
    estimativas_extra = resultados[N_extra]['estimativas']
    # Ajuste de uma Gaussiana aos dados (média e desvio)
    mu, sigma = norm.fit(estimativas_extra)
    
    # Criação do histograma
    plt.figure(figsize=(12, 7))
    n, bins, patches = plt.hist(estimativas_extra, bins=50, density=True,
                                alpha=0.6, color='skyblue', edgecolor='black',
                                label='Histograma das estimativas')
    
    # Curva da Gaussiana ajustada
    x = np.linspace(min(estimativas_extra), max(estimativas_extra), 200)
    pdf = norm.pdf(x, mu, sigma)
    plt.plot(x, pdf, 'r-', linewidth=2, label=f'Gaussiana ajustada\nμ = {mu:.6f}, σ = {sigma:.6f}')
    
    # Linha vertical indicando o valor real de π
    plt.axvline(np.pi, color='green', linestyle='--', linewidth=1.5, label=f'π real = {np.pi:.6f}')
    
    plt.title(f'Distribuição das estimativas de π para N = {N_extra} lançamentos (M = {resultados[N_extra]["M"]} repetições)')
    plt.xlabel('Estimativa de π')
    plt.ylabel('Densidade de probabilidade')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"  Média ajustada (μ) = {mu:.6f}")
    print(f"  Desvio padrão (σ)  = {sigma:.6f}")
else:
    print("N = 10000 não encontrado nos resultados; verifique a configuração.")

